In [ ]:
import os, sys
# Resolve code/beh_ephys_analysis (the folder containing `utils`) relative to this
# file's location, so imports work no matter where the repo is checked out.
_anchor = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.path.abspath(os.getcwd())
while _anchor != os.path.dirname(_anchor):
    _beh_ephys_root = os.path.join(_anchor, "code", "beh_ephys_analysis")
    if os.path.isdir(os.path.join(_beh_ephys_root, "utils")):
        if _beh_ephys_root in sys.path:
            sys.path.remove(_beh_ephys_root)
        sys.path.insert(0, _beh_ephys_root)
        break
    _anchor = os.path.dirname(_anchor)
from utils.capsule_migration import CAPSULE_ROOT

In [ ]:
from behavior_and_time_alignment import beh_and_time_alignment
from session_preprocessing import ephys_opto_preprocessing, ephys_opto_crosscorr
from opto_sigs import cal_opto_sigs
from opto_tagging import opto_plotting_session
from opto_waveforms_preprocessing import opto_wf_preprocessing, re_filter_opto_waveforms, go_cue_waveforms
from opto_tagging_summary import opto_tagging_summary
from cross_auto_corr import cross_auto_corr, plot_cross_auto_corr
from drift_analysis import plot_session_opto_drift, generate_session_opto_drift_trial_table
from antidromic_analysis_session import plot_opto_responses_session, analyze_antidromic_responses
from plot_waveform_quality import waveform_check, compare_mean_go_cue_waveforms
import matplotlib.pyplot as plt
from unit_beh_analysis import plot_unit_beh_session, plot_alignments
from pupil_analysis import plot_unit_pupil_correlation, pupil_analysis_session
import pandas as pd

In [ ]:
session_assets = pd.read_csv(CAPSULE_ROOT + '/code/data_management/session_assets.csv')
session_list = session_assets['session_id'].tolist()

In [ ]:
# example session
session = 'behavior_754897_2025-03-14_11-28-53'

# Session Processing Workflow
This notebook runs opto/ephys preprocessing and analysis for one session or a batch of sessions.

## Modes
- **`data_type='raw'`**: use this when data is not curated yet. This mode performs behavior/time alignment first, then ephys/opto preprocessing and tagging analysis for visual inspection and curation.
- **`data_type='curated'`**: use this after merge/split/relabel curation. This reruns processing on curated data and skips the initial alignment check.
- **`post_drift_correction=False`**: drift-update mode; applies drift cut points and reruns correlation outputs.
- **`behavior=False`**: whether to perform analysis on ephys data in behavior
- **`pupil=False`**: whether to perform analysis on pupil data in behavior and against ephys data. 

## Detailed Pipeline
### 1) Raw-only alignment
- `beh_and_time_alignment`: checks alignment between behavior, sound, and ephys streams.

### 2) Core preprocessing (raw and curated)
- `ephys_opto_preprocessing`: prepares spiketimes/opto tables and core opto response structures.

### 3) Curated-only preprocessing before plotting
1. `opto_wf_preprocessing`
   - Computes waveform similarity between laser-triggered and spontaneous spikes.
   - Helps identify potential artifacts/misclassified units during curation.
2. `ephys_opto_crosscorr`
   - Computes auto/cross-correlation for opto-tagged and non-tagged units on short (20 ms) and long (1 s) scales.
   - Useful for checking ISI-violation-like structure and correlation changes by tagging status.
3. `re_filter_opto_waveforms`
   - Recomputes waveforms with a filter closer to raw (`50-8000 Hz`).

### 4) Opto-tagging analysis
- `opto_plotting_session`: plots laser-triggered response summaries.
- `cal_opto_sigs`: computes fraction of stimulations inducing significant responses per unit.

### 5) Antidromic tagging analysis
- `plot_opto_responses_session`: visualizes laser-triggered responses for antidromic tagging.
- `analyze_antidromic_responses`: computes antidromic statistics and tiering.

### 6) Curated-only post-plot summaries
1. `opto_tagging_summary`
   - Spatial summary of opto responses and response-property overview.
2. `plot_session_opto_drift`
   - Detects possible drift and suggests candidate cut points.
3. `generate_session_opto_drift_trial_table`
   - Creates trial-level drift annotations for downstream analyses.

### 7) Waveform quality check
- `waveform_check`
  - Plots waveform mean ± SD/SEM and waveform distribution across bins.
  - Displays raw traces for randomly sampled spikes on major channels.
  - Helps identify artifacts, low-SNR units

### 8) Go-cue aligned waveforms
- `go_cue_waveforms`
  - Extracts spikes aligned to go-cue events (sorted by behavior trial alignment).
  - Computes raw-filtered and bandpass-filtered waveforms around go-cue times.
  - This is to check if electric signals distorted waveforms.
- `compare_mean_go_cue_waveforms`
  - Plot mean waveforms of spikes around time of go cue vs the template.

 
### 9) After manual drift inspection/correction
- Set `post_drift_correction=True` to apply cut points and rerun:
  1. `cross_auto_corr`: recompute auto/cross-correlation for spontaneous and behavior epochs.
  2. `plot_cross_auto_corr`: plot the recomputed correlation outputs.

### 10) Optional behavior-linked unit analysis (`behavior=True`)
- `plot_unit_beh_session` aligned to:
  - `response`
  - `go_cue`
- `plot_alignments`: plot neuron activity around go cue and predict neuron activity based on history

### 11) Optional pupil analysis with behavior and units
- `pupil_analysis_session` analysis pupil dynamics in behavior task
- `plot_unit_pupil_correlation` compute and plot cross-correlation between pupil and neurons.

## Notes
- Session-level failures in batch mode are logged and processing continues.
- Keep boolean flags distinct from function names (for example `plot_session_opto_drift_flag`) to avoid shadowing.

In [ ]:
# Processing configuration
data_type = 'curated'  # 'curated' or 'raw'
behavior = False  # Run behavior-linked unit analysis at the end.
pupil = False  # Run pupil-linked unit analysis at the end.
plot_session_opto_drift_flag = False  # Run drift-update-only mode.

def process(session, data_type='raw', behavior=False, pupil=False, post_drift_correction=False):
    """Run preprocessing and analysis for one session.

    Args:
        session: Session id like 'behavior_754897_2025-03-14_11-28-53'.
        data_type: 'raw' or 'curated'.
        behavior: If True, run behavior-linked unit activity analysis.
        pupil: If True, run pupil-linked analysis for behavior and units.
        post_drift_correction: If True, only update drift cut points and rerun
            cross/auto-correlation outputs.
    """
    def log(msg):
        print(f'[{session}] {msg}')

    plt.close('all')
    log(f'start (data_type={data_type}, behavior={behavior}, pupil={pupil}, post_drift_correction={post_drift_correction})')

    if post_drift_correction:
        log('drift-update mode: updating drift cut points')
        plot_session_opto_drift(session, data_type, update_csv=True, plot=True, update_cut=True)
        log('drift-update mode: recomputing cross/auto-correlation')
        cross_auto_corr(session, data_type)
        plot_cross_auto_corr(session, data_type)
        log('finished drift-update mode')
        return

    if data_type == 'raw':
        # Alignment is needed before downstream raw preprocessing.
        log('running behavior/time alignment (raw mode)')
        beh_and_time_alignment(session)
        plt.close('all')
        log('completed behavior/time alignment')

    # Core preprocessing for both raw and curated data.
    log('running core opto/ephys preprocessing')
    ephys_opto_preprocessing(session, data_type, 'soma')
    log('completed core opto/ephys preprocessing')

    if data_type == 'curated':
        # Curated-only waveform cleanup and correlation diagnostics.
        log('running curated waveform preprocessing')
        opto_wf_preprocessing(session, data_type, 'soma', load_sorting_analyzer=True)
        log('running curated opto cross-correlation')
        ephys_opto_crosscorr(session, data_type)
        log('running curated waveform re-filtering')
        re_filter_opto_waveforms(session, data_type, opto_only=True, load_sorting_analyzer=True)
        log('completed curated-only preprocessing')

    # Opto-tagging analysis.
    log('running opto-tagging plotting and significance analysis')
    opto_plotting_session(session, data_type, 'soma', plot=True, resp_thresh=0.3, lat_thresh=0.02, save=True)
    cal_opto_sigs(session, data_type, 'soma', plot=True, save=True)
    log('completed opto-tagging analysis')

    # Antidromic tagging analysis.
    log('running antidromic response plotting/statistics')
    plot_opto_responses_session(session, data_type=data_type, opto_only=True)
    analyze_antidromic_responses(session, data_type=data_type, plot=False, tier_cat=True)
    log('completed antidromic analysis')

    if data_type == 'curated':
        log('running curated summaries')
        opto_tagging_summary(session, data_type, 'soma', plot=True, save=True)

        # Initial drift analysis (without applying new cut points yet).
        log('running initial drift analysis (update_cut=False)')
        plot_session_opto_drift(session, data_type, update_csv=True, plot=True, update_cut=False)
        generate_session_opto_drift_trial_table(session, data_type, opto_only=False, save=True)
        log('completed curated summaries and drift table generation')

        # Waveform quality check (step 7).
        log('running waveform quality check')
        waveform_check(session, data_type=data_type, opto_only=True)
        log('completed waveform quality check')

        # Go-cue waveforms (step 8).
        log('running go-cue waveforms analysis')
        go_cue_waveforms(session, data_type, opto_only=True, load_sorting_analyzer=True)
        compare_mean_go_cue_waveforms(session, data_type)
        log('completed go-cue waveforms analysis')

    if behavior:
        log('running behavior-linked unit analysis')
        model_name = 'stan_qLearning_5params'
        curate_time = False
        formula = 'spikes ~ 1 + outcome + choice + Qchosen'

        plot_unit_beh_session(
            session,
            data_type=data_type,
            align_name='response',
            curate_time=curate_time,
            model_name=model_name,
            formula=formula,
            pre_event=-1,
            post_event=3,
            binSize=0.3,
            stepSize=0.1,
            units=None,
            opto_only=True,
        )

        plot_unit_beh_session(
            session,
            data_type=data_type,
            align_name='go_cue',
            curate_time=curate_time,
            model_name=model_name,
            formula=formula,
            pre_event=-1,
            post_event=3,
            binSize=0.3,
            stepSize=0.1,
            units=None,
            opto_only=True,
        )

        plot_alignments(session, data_type=data_type)
        log('completed behavior-linked analysis')

    if pupil:
        log('running pupil-linked unit analysis')
        pupil_analysis_session(session, plot_licks=True, plot=True)
        plot_unit_pupil_correlation(session, bin_size=5, step_size=0.1, win_length=20, plot=True)
        log('completed pupil-linked analysis')

    log('finished')

In [ ]:
# Example session run
process(
    session,
    data_type=data_type,
    behavior=behavior,
    pupil=pupil,
    post_drift_correction=plot_session_opto_drift_flag,
)

In [ ]:
# Sequential processing of selected sessions
for session in session_list:
    try:
        process(
            session,
            data_type=data_type,
            behavior=behavior,
            pupil=pupil,
            post_drift_correction=plot_session_opto_drift_flag,
        )
    except Exception as exc:
        print(f'Failed to process {session}: {exc}')